# Example notebook for registering a new circuit
- Checks metadata provided
- Registers circuit entity
- Generates and uploads assets

## Circuit information

### General metadata

To be provided as a dict with key/value pairs. Optional entries can be set to None.

**IMPORTANT:**
- **The circuit name must not yet exist in entitycore (to avoid duplicates)**

- **Linked entities must exist in entitycore. Otherwise, they must first be registered:**
  - Species
  - Subject
  - Brain region hierarchy
  - Brain region
  - License


In [ ]:
circuit_metadata = {
    # Name of the circuit
    "name": "EXAMPLE_01__N_10__top_nodes_dim6",

    # Description of the circuit
    "description": "Example circuit with 10 neurons",
    
    # Circuit build category (computational_model, em_reconstruction)
    "build_category": "computational_model",

    # Species name [Must exist in entitycore; otherwise, must first be registered]
    "species": "Rattus norvegicus",

    # Subject name [Must exist in entitycore; otherwise, must first be registered]
    "subject": "Average rat P14",

    # Brain region hierarchy name for the given species [Must exist in entitycore; otherwise, must first be registered]
    "brain_region_hierarchy": "Waxholm Space atlas of the rat brain with layers and hippocampal subregions",

    # Brain region name within hierarchy [Must exist in entitycore; otherwise, must first be registered]
    "brain_region": "Primary somatosensory area, hindlimb representation",

    # Optional: Name of the root circuit (= top parent in the extraction/derivation hierarchy), if any
    "root": None,

    # Optional: Name of the parent circuit (=direct parent the circuit was extracted or derived from)
    "parent": None,

    # Optional: Type of circuit derivation (circuit_extraction, circuit_rewiring), if parent provided
    "derivation_type": None,

    # Optional: License name [Must exist in entitycore; otherwise, must first be registered]
    "license": "CC BY 4.0",

    # Optional: Short, human readable publication string
    "published_in": "Reimann et al and Isbister et al",

    # Optional: Contact e-mail address
    "contact": "support@openbraininstitute.org",

    # Optional: Experiment date (=date when the circuit has been build or published)
    # Note: Two date formats supported: "November, 2024" (treated as 01.11.2024) and "27.03.2024"
    "experiment_date": "15.05.2025",
    }

### Circuit path

Path to the SONATA circuit folder containing the `circuit_config.json`.

In [ ]:
# SONATA circuit folder with circuit_config.json
circuit_path = "../data/tiny_circuits/N_10__top_nodes_dim6"

### Optional: Contributors

To be provided as a dict:
- Key = Agent name **[Must exist in entitycore; otherwise, must first be registered]**
- Value = Dict with
  - "type" (person, organization, consortium)
  - "role" (unspecified, ...)

In [ ]:
circuit_contributions = {
    # Contributing persons
    "Christoph Pokorny": {"type": "person", "role": "unspecified"},
    # "<ADD NAME>": {"type": "person", "role": "unspecified"},

    # Contributing organizations
    "Open Brain Institute, Lausanne, Switzerland": {"type": "organization", "role": "unspecified"},
    # "<ADD NAME>": {"type": "organization", "role": "unspecified"},

    # Contributing consortia
    # "<ADD NAME>": {"type": "consortium", "role": "unspecified"},
 }

### Optional: Publications

To be provided as a dict:
- Key = DOI **[Must exist in entitycore; otherwise, must first be registered]**
- Value = Dict with
  - "type" (entity_source, component_source, application)

In [ ]:
circuit_publications = {
    # Circuit source publication(s)
    "10.7554/eLife.99688.2": {"type": "entity_source"},
    "10.7554/eLife.99693.2": {"type": "entity_source"},

    # Component source publication(s)
    # "<ADD DOI>": {"type": "component_source"},

    # Circuit application(s)
    # "<ADD DOI>": {"type": "application"},
    }

## Get DB client

Specify environment (staging, production) and project context.

In [ ]:
from entitysdk import Client, ProjectContext, models
from obi_auth import get_token

In [ ]:
# "Shared results + tests"
token = get_token(environment="staging")
project_context = ProjectContext(virtual_lab_id="e6030ed8-a589-4be2-80a6-f975406eb1f6", project_id="2720f785-a3a2-4472-969d-19a53891c817")
client = Client(environment="staging", project_context=project_context, token_manager=token)


## Run circuit registration

First, a dry run (`dry_run = True`) should be used to check the circuit files, metadata, and dependencies but w/o registration to entitycore:
- Valid SONATA circuit with circuit_config.json must exist
- Additional assets will be generated locally
- The circuit name must not yet exist in entitycore (i.e., to avoid duplicates)
- All dependencies must already exist in entitycore (i.e., must be registered beforehand)

If successful, the actual registration can be done by re-running with `dry_run = False`.

The circuit can be made public by setting `authorized_public = True`, otherwise it will be private within the chosen project.

In [ ]:
from obi_one.utils.circuit_registration import register_circuit_from_metadata

In [ ]:
dry_run = True  # If True, runs all checks but no actual registration; otherwise actual registration
authorized_public = False  # If True, registered as public entity; otherwise private within project

In [ ]:
# Create & register circuit entity
registered_circuit = register_circuit_from_metadata(
    client=client,
    circuit_metadata=circuit_metadata,
    circuit_path=circuit_path,
    contributions=circuit_contributions,
    publications=circuit_publications,
    authorized_public=authorized_public,
    dry_run=dry_run,
)

After registration, the registered Circuit entity can be checked.

In [ ]:
if registered_circuit:
    # Fetch entity incl. assets
    res = client.get_entity(entity_id=registered_circuit.id, entity_type=models.Circuit)
    
    # Print circuit info
    print(f"Circuit '{res.name}' registered (ID {res.id})")
    print(f"  scale={res.scale}, neurons={res.number_neurons}, synapses={res.number_synapses}, connections={res.number_connections}")
    print(f"  has_morphologies={res.has_morphologies}, has_point_neurons={res.has_point_neurons}, has_electrical_cell_models={res.has_electrical_cell_models}, has_spines={res.has_spines}")

    # Print asset info
    print(f"Assets registered ({len(res.assets)}):")
    for a in res.assets:
        print(f"  Asset '{a.label}' ({'dir' if a.is_directory else 'file'}) registered (ID {a.id})")
else:
    print("Circuit not registered!")